Football Player Future G/A Predictor

Importing Modules

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

Reading the Data

In [2]:
df23 = pd.read_csv('data/football-data_23-24.csv') 
df24 = pd.read_csv('data/football-data_24-25.csv')
df25 = pd.read_csv('data/football-data_25-26.csv')

print(f"2023-2024 shape: {df23.shape}")
print(f"2024-2025 shape: {df24.shape}")
print(f"2025-2026 shape: {df25.shape}")

2023-2024 shape: (2852, 37)
2024-2025 shape: (2854, 267)
2025-2026 shape: (2779, 102)


Preparing and Cleaning the data

In [3]:
base_features = ['Min', '90s', 'Gls', 'Ast', 'G+A', 'xG', 'npxG', 'xAG', 'PrgC', 'PrgP', 'PrgR']

df23_sub = df23[['Player', 'Age', 'Pos'] + base_features]
df23_sub = df23_sub.add_suffix('_23')
df23_sub = df23_sub.rename(columns={'Player_23': 'Player'})

df24_sub = df24[['Player'] + base_features]
df24_sub = df24_sub.add_suffix('_24')
df24_sub = df24_sub.rename(columns={'Player_24': 'Player'})

df25_target = df25[['Player', 'G+A']].rename(columns={'G+A': 'Target_G+A'})

cleandf = df23_sub.merge(df24_sub, on='Player', how='inner').merge(df25_target, on='Player', how='inner')
cleandf = cleandf.drop_duplicates(subset=['Player'])

cleandf = cleandf[(cleandf['90s_24'] >= 10) & (cleandf['90s_23'] >= 5)].copy()
cleandf['Age_current'] = cleandf['Age_23'].astype(float) + 2

for season in ['23', '24']:
    cleandf[f'GA_per90_{season}']  = cleandf[f'G+A_{season}']  / cleandf[f'90s_{season}']
    cleandf[f'xG_per90_{season}']  = cleandf[f'xG_{season}']   / cleandf[f'90s_{season}']
    cleandf[f'xAG_per90_{season}'] = cleandf[f'xAG_{season}']  / cleandf[f'90s_{season}']

# How consistent is the player across both seasons and what is the variation (volatility) - predictability
cleandf['GA_consistency'] = (cleandf['GA_per90_24'] + cleandf['GA_per90_23']) / 2
cleandf['GA_volatility'] = abs(cleandf['GA_per90_24'] - cleandf['GA_per90_23'])
cleandf['xG_consistency']  = (cleandf['xG_per90_24']  + cleandf['xG_per90_23'])  / 2
cleandf['xG_volatility']   = abs(cleandf['xG_per90_24']  - cleandf['xG_per90_23'])
cleandf['xAG_consistency'] = (cleandf['xAG_per90_24'] + cleandf['xAG_per90_23']) / 2
cleandf['xAG_volatility']  = abs(cleandf['xAG_per90_24'] - cleandf['xAG_per90_23'])

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cleandf['Pos_encoded'] = le.fit_transform(cleandf['Pos_23'].fillna('Unknown'))

print(f"Players after filter: {cleandf.shape[0]}")


Players after filter: 921


Training the model using the test data (Random Forest Regressor)

In [4]:
features = ['Age_current', 'Pos_encoded', '90s_23', '90s_24','GA_per90_23', 'xG_per90_23', 'xAG_per90_23',
'GA_per90_24', 'xG_per90_24', 'xAG_per90_24','GA_consistency', 'GA_volatility','xG_consistency', 'xG_volatility',
'xAG_consistency', 'xAG_volatility','PrgC_24', 'PrgP_24', 'PrgR_24']

X = cleandf[features].fillna(0)
y = cleandf['Target_G+A'].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"Model Accuracy (MAE): {mae:.3f} Goals/Assists off on average\n")

results = cleandf.loc[X_test.index, ['Player', 'Pos_23', 'Target_G+A']].copy()
results['Predicted_G+A'] = y_pred

top_predictions = results.sort_values(by='Predicted_G+A', ascending=False)

print("🏆 Model's Top 10 Predictions for the 25/26 Season (Test Set):")
print(top_predictions.head(10).to_string(index=False))

Model Accuracy (MAE): 2.471 Goals/Assists off on average

🏆 Model's Top 10 Predictions for the 25/26 Season (Test Set):
         Player Pos_23  Target_G+A  Predicted_G+A
Serhou Guirassy     FW          17      19.536908
 Alexander Isak     FW           4      13.555892
Mason Greenwood  MF,FW          21      13.483940
 Iñaki Williams     FW           6      12.365614
     Alex Baena  MF,FW           5      11.754328
   Amine Gouiri  FW,MF          10      10.745700
Nicolas Jackson     FW           8      10.333185
Jude Bellingham     MF           8      10.329700
Vinicius Júnior     FW          20       9.859831
   Ante Budimir     FW          17       9.694570


Final Predictions using the entire datset

In [5]:
all_predictions = rf_model.predict(X)
global_results = cleandf[['Player', 'Pos_23', 'Target_G+A']].copy()

mae2 = mean_absolute_error(y, all_predictions)
print(f"Model Accuracy (MAE2): {mae2:.3f} Goals/Assists off on average\n")

global_results['Predicted_G+A'] = all_predictions
global_top = global_results.sort_values(by='Predicted_G+A', ascending=False)

print("🌍 GLOBAL TOP 10 PREDICTIONS (Entire Dataset):")
print(global_top.head(20).to_string(index=False))

Model Accuracy (MAE2): 1.800 Goals/Assists off on average

🌍 GLOBAL TOP 10 PREDICTIONS (Entire Dataset):
            Player Pos_23  Target_G+A  Predicted_G+A
        Harry Kane     FW          38      31.247143
     Kylian Mbappé     FW          28      25.947872
     Michael Olise  FW,MF          33      24.660755
    Erling Haaland     FW          32      24.309743
      Lamine Yamal     FW          27      21.249296
   Bruno Fernandes  MF,FW          27      20.680468
   Serhou Guirassy     FW          17      19.536908
         Luis Díaz     FW          28      19.014979
       Deniz Undav  FW,MF          24      17.536587
     Mohamed Salah     FW          13      17.441264
     Patrik Schick     FW          19      15.619794
   Ousmane Dembélé  MF,FW          16      15.524248
       Cole Palmer  FW,MF          10      14.672013
          Raphinha  FW,MF          14      14.433286
Robert Lewandowski     FW          15      14.325513
     Ferrán Torres     FW          16      14.1

Using the Gradient Boosting Regressor

In [6]:
from sklearn.ensemble import GradientBoostingRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, y_pred_gb)
print(f"Gradient Boosting Accuracy (Test MAE): {gb_mae:.3f} Goals/Assists off on average\n")

all_predictions_gb = gb_model.predict(X)
global_results_gb = cleandf[['Player', 'Pos_23', 'Target_G+A']].copy()

global_results_gb['Predicted_G+A'] = all_predictions_gb

global_top_gb = global_results_gb.sort_values(by='Predicted_G+A', ascending=False)

print("🚀 GLOBAL TOP 10 PREDICTIONS (Gradient Boosting):")
print(global_top_gb.head(30).to_string(index=False))

Gradient Boosting Accuracy (Test MAE): 2.560 Goals/Assists off on average

🚀 GLOBAL TOP 10 PREDICTIONS (Gradient Boosting):
            Player Pos_23  Target_G+A  Predicted_G+A
        Harry Kane     FW          38      36.733725
     Michael Olise  FW,MF          33      32.292045
    Erling Haaland     FW          32      29.462385
     Kylian Mbappé     FW          28      29.181728
      Lamine Yamal     FW          27      25.348539
   Bruno Fernandes  MF,FW          27      24.166217
         Luis Díaz     FW          28      22.093945
       Deniz Undav  FW,MF          24      20.136926
   Serhou Guirassy     FW          17      18.246224
     Patrik Schick     FW          19      17.140919
      Vedat Muriqi     FW          22      16.777051
   Andrej Kramarić  MF,FW          19      16.652946
    Alexander Isak     FW           4      16.415067
  Lautaro Martínez     FW          21      16.347276
   Ousmane Dembélé  MF,FW          16      16.038360
Georges Mikautadze     FW   